# SPICEM — Human Kidney Cohort Analysis

## Output folders

| Folder | Content | Metabolite selection | Row ranking |
|---|---|---|---|
| `plots_consensus_cohort/` | Violin + dot + donut + clinical strip | CKD-specificity + clinical boost | — |
| `merged_plots_combined/` | Multi-row streamline, 3-group panels | Same as consensus | W_max × \|Δcons\| × stat |
| `cohort_comparison_wceg/` | WCEG volcano plots + CSVs | All detected | WCEG permutation |
| `merged_plots_wceg_sig/` | Streamlines for WCEG-significant mets | Sig hits from volcano | WCEG W weight |

**Key**: `merged_plots_wceg_sig/` is the direct visual companion to the volcano plots —
the same metabolites, shown spatially with their top WCEG-weighted exchange rows.


## 0. Setup

In [ ]:
import sys, os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath("/path/to/spicem")   # EDIT ME
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"REPO_ROOT: {REPO_ROOT}")


## 1. Configuration

In [ ]:
DATA_ROOT  = "/path/to/all_patient_data"   # EDIT ME
COHORT_OUT = "/path/to/all_patient_results"   # EDIT ME
os.makedirs(COHORT_OUT, exist_ok=True)

REGIONS          = [1]
NUM_WORKERS      = 7
PERMUTATION_ITER = 500
GRID_RES         = 80
GAUSSIAN_SIGMA   = 1.2
TOP_N_PLOTS      = 20
RANDOM_SEED      = 42

# Consensus / combined plot settings
COHORT_CONSENSUS_TOP_N = 20
CLINICAL_FIELDS        = ["eGFR", "fibrosis"]
CLINICAL_WEIGHT        = 0.40
CKD_WEIGHT             = 0.60
TOP_CLINICAL_K         = 5

# Merged plot settings
MERGED_TOP_EXCHANGES  = 5
COMBINED_P_FLOOR      = 0.5

# Clinical correlation settings
PADJ_THRESHOLD        = 0.20   # BH-FDR threshold (liberal for n=14)

# WCEG settings
WCEG_N_PERMUTATIONS = 1000
WCEG_ALPHA          = 0.20

# Group comparison
USE_CORRECTED_FOR_COMPARISON = False

PATIENTS = None
print("Config OK")


## 2. Run full pipeline

In [ ]:
from run_cohort_pipeline import run_cohort, CohortConfig

cohort_cfg = CohortConfig(
    data_root=DATA_ROOT, base_out_dir=COHORT_OUT,
    patients=PATIENTS, regions=REGIONS,
    num_workers=NUM_WORKERS, permutation_iters=PERMUTATION_ITER,
    grid_resolution=GRID_RES, gaussian_sigma=GAUSSIAN_SIGMA,
    top_n_plots=TOP_N_PLOTS, random_seed=RANDOM_SEED,
    skip_existing=True,
    # Consensus summary plots
    run_cohort_consensus=True,
    cohort_consensus_top_n=COHORT_CONSENSUS_TOP_N,
    clinical_fields=CLINICAL_FIELDS,
    clinical_weight=CLINICAL_WEIGHT,
    ckd_weight=CKD_WEIGHT,
    top_clinical_k=TOP_CLINICAL_K,
    # Merged combined plots (CKD + clinical mets)
    run_merged_plots_combined=True,
    merged_top_exchanges=MERGED_TOP_EXCHANGES,
    merged_combined_p_floor=COMBINED_P_FLOOR,
    # WCEG
    wceg_n_permutations=WCEG_N_PERMUTATIONS,
    wceg_alpha=WCEG_ALPHA,
    use_corrected_scores_for_comparison=USE_CORRECTED_FOR_COMPARISON,
    use_wceg_comparison=True,
    verbose=True,
)
summary_df = run_cohort(cfg=cohort_cfg)
summary_df


## 3. Standalone re-generation

### 3.1 Load patient results

In [ ]:
from run_cohort_pipeline import load_patient_results
from consensus_exchange_network import build_exchange_records

patient_results = load_patient_results(COHORT_OUT)
exchange_df     = build_exchange_records(patient_results)
print(f"Patients: {len(patient_results)}")
print(f"Exchange records: {exchange_df.shape}  "
      f"({exchange_df['metabolite'].nunique()} mets, "
      f"{exchange_df['patient_id'].nunique()} patients)")


### 3.2 Consensus summary plots

In [ ]:
from cohort_consensus_plots import generate_cohort_consensus_plots

consensus_out = os.path.join(COHORT_OUT, "plots_consensus_cohort")
saved, selected_mets, rank_df = generate_cohort_consensus_plots(
    patient_results, out_dir=consensus_out,
    top_n=COHORT_CONSENSUS_TOP_N,
    clinical_fields=CLINICAL_FIELDS,
    clinical_weight=CLINICAL_WEIGHT,
    ckd_weight=CKD_WEIGHT,
    top_clinical_k=TOP_CLINICAL_K,
    verbose=True,
)
print(f"{len(saved)} figures  |  {len(selected_mets)} metabolites selected")


### 3.3 Merged combined plots

In [ ]:
from merged_streamline_plots import generate_merged_streamline_plots_combined

combined_out = os.path.join(COHORT_OUT, "merged_plots_combined")
saved_c = generate_merged_streamline_plots_combined(
    patient_results,
    out_dir=combined_out,
    top_n=COHORT_CONSENSUS_TOP_N,
    clinical_fields=CLINICAL_FIELDS,
    clinical_weight=CLINICAL_WEIGHT,
    ckd_weight=CKD_WEIGHT,
    top_clinical_k=TOP_CLINICAL_K,
    top_exchanges=MERGED_TOP_EXCHANGES,
    p_floor=COMBINED_P_FLOOR,
    wceg_n_permutations=WCEG_N_PERMUTATIONS,
    wceg_alpha=WCEG_ALPHA,
    exchange_df=exchange_df,
    verbose=True,
)
print(f"{len(saved_c)} combined figures -> {combined_out}")


### 3.4 Preview combined plots

In [ ]:
from IPython.display import Image, display
figs = sorted(glob.glob(os.path.join(combined_out, "*.png")))
print(f"{len(figs)} figures")
for fp in figs[:3]:
    print(os.path.basename(fp))
    display(Image(fp, width=1400))


In [ ]:
import sys

if 'cohort_comparison' in sys.modules:
    del sys.modules['cohort_comparison']

## 4. Group comparisons

In [ ]:
from cohort_comparison import ComparisonConfig, run_all_comparisons

comp_cfg = ComparisonConfig(
    base_out_dir=COHORT_OUT,
    n_permutations=5000,
    perm_seed=RANDOM_SEED,
    p_thresh=0.05,
    #use_corrected_scores=USE_CORRECTED_FOR_COMPARISON,
    #use_wceg_comparison=True,
    #wceg_n_permutations=WCEG_N_PERMUTATIONS,
    #wceg_alpha=WCEG_ALPHA,
    verbose=True,
)
comparison_results = run_all_comparisons(
    patient_results, comp_cfg)
print(f"\nCompleted {len(comparison_results)} comparisons")


### 4.1 Preview volcano plots

In [ ]:
from IPython.display import Image, display
volc_dir = os.path.join(COHORT_OUT, "group_comparisons")
for fp in sorted(glob.glob(os.path.join(volc_dir, "*volcano_*.png"))):
    print(os.path.basename(fp))
    display(Image(fp, width=1050))

### 4.2 Significant hits summary

In [ ]:
for (ga, gb), df in comparison_results.items():
    if df.empty: continue
    n_sig  = int(df["significant"].sum())
    n_tot  = len(df)
    if n_sig == 0: continue
    print(f"\n{ga} vs {gb}: {n_sig}/{n_tot} significant edges")
    cols = ["metabolite","source","sink","log2fc","evidence_score"]
    cols = [c for c in cols if c in df.columns]
    top  = df[df["significant"]].nlargest(10,"evidence_score")[cols]
    print(top.to_string(index=False))


## 5. Merged streamline plots — WCEG-significant metabolites

Generates spatial streamline figures for the metabolites that are statistically
significant in the WCEG volcano comparisons. This is the direct visual-statistical
bridge: the same metabolites appear in the volcano AND the spatial plots.

Each comparison gets its own sub-folder under `merged_plots_wceg_sig/`.
Exchange rows within each figure are ranked by W = mean(S_corr) × consistency.


In [ ]:
from merged_streamline_plots import generate_merged_streamline_plots_wceg_sig

sig_out = os.path.join(COHORT_OUT, "merged_plots_wceg_sig")
saved_ws = generate_merged_streamline_plots_wceg_sig(
    patient_results,
    comparison_results=comparison_results,
    out_dir=sig_out,
    wceg_n_permutations=WCEG_N_PERMUTATIONS,
    wceg_alpha=WCEG_ALPHA,
    exchange_df=exchange_df,
    top_metabolites=20,
    top_exchanges=MERGED_TOP_EXCHANGES,
    verbose=True,
)
for comp_label, paths in saved_ws.items():
    print(f"  {comp_label}: {len(paths)} figures")


### 5.1 Preview

In [ ]:
from IPython.display import Image, display
for comp_label in ["DKD_vs_Control","HKD_vs_Control","DKD_vs_HKD"]:
    folder = os.path.join(sig_out, comp_label)
    figs   = sorted(glob.glob(os.path.join(folder, "*.png")))
    if not figs: continue
    print(f"\n── {comp_label}: {len(figs)} figures ──")
    for fp in figs[:3]:
        print(os.path.basename(fp))
        display(Image(fp, width=1400))


In [ ]:
import sys
if 'merged_streamline_plots' in sys.modules:
    del sys.modules['merged_streamline_plots']

In [ ]:
import sys
if 'run_cohort_pipeline' in sys.modules:
    del sys.modules['run_cohort_pipeline']

## 6. Merged streamline plots — clinical correlation

Uses **Spearman correlation + Benjamini-Hochberg FDR** across all 14 patients
to identify exchanges that track clinical disease severity.

**Why this is complementary to the WCEG volcano:**

| Approach | Captures | Limitation |
|---|---|---|
| WCEG volcano (group) | Exchanges different between DKD and Control | n=3 Control limits power |
| Clinical correlation | Exchanges that scale with eGFR/fibrosis continuously | No group distinction |

A metabolite significant in both is the strongest finding.
A metabolite only in clinical correlation means it tracks disease severity
regardless of diagnostic label — potentially more biologically meaningful.

**Statistical approach:**
1. For each (metabolite, src→snk) edge: score vector = mean S_corr per patient (0 if not detected)
2. Spearman ρ with each clinical variable (eGFR, fibrosis, age, sex, hypertension, diabetes)
3. Benjamini-Hochberg FDR correction across all edges per variable
4. Select metabolites at padj < 0.20 (relaxed to 0.50 if fewer than 5 pass)
5. Rank by W_max × |ρ| × (−log10 padj + floor)


In [ ]:
from merged_streamline_plots import generate_merged_streamline_plots_clinical

clinical_out = os.path.join(COHORT_OUT, "merged_plots_clinical")
saved_cl = generate_merged_streamline_plots_clinical(
    patient_results,
    out_dir=clinical_out,
    #clinical_fields=CLINICAL_FIELDS + ["age"],
    clinical_fields = ["eGFR", "fibrosis"],
    #padj_threshold=PADJ_THRESHOLD,
    #top_metabolites=COHORT_CONSENSUS_TOP_N,
    top_metabolites = 50,
    top_exchanges=MERGED_TOP_EXCHANGES,
    p_floor=COMBINED_P_FLOOR,
    exchange_df=exchange_df,
    verbose=True,
)
print(f"\n{len(saved_cl)} clinical-correlation figures -> {clinical_out}")


### 6.1 Clinical correlation table

In [ ]:
corr_table = pd.read_csv(os.path.join(clinical_out, "clinical_correlations.csv"))
print(f"Total edges: {len(corr_table)}")
print(f"Significant (padj<{PADJ_THRESHOLD}): "
      f"{(corr_table['best_padj']<PADJ_THRESHOLD).sum()}")
print("\nTop 20 by signal:")
cols = ["metabolite","source","sink","best_field","best_rho","best_padj","signal","W_max"]
cols = [c for c in cols if c in corr_table.columns]
print(corr_table[cols].head(500).to_string(index=False))


### 6.2 Preview clinical plots

In [ ]:
from IPython.display import Image, display
figs = sorted(glob.glob(os.path.join(clinical_out, "*.png")))
print(f"{len(figs)} figures")
for fp in figs[:4]:
    print(os.path.basename(fp))
    display(Image(fp, width=1400))


## 7. Clinical association statistics

Runs all clinical analyses from `cohort_statistics.py`:

| Analysis | Method | Output |
|---|---|---|
| 1. eGFR correlation | Spearman ρ across 14 patients | Ranked bar chart, scatter plots, cell-type strip |
| 2. Fibrosis association | Spearman ρ | Ranked bar chart, scatter plots, box plots by tertile |
| 3. Sex association | Mann-Whitney U | Volcano plot |
| 4. Hypertension association | Mann-Whitney U | Volcano plot |
| 5. Diabetes association | Mann-Whitney U | Volcano plot |
| 6. Multi-variable regression | OLS: score ~ eGFR + fibrosis + age + sex + is_DKD + is_HKD | β-coefficient heatmap, CSV |
| 7. Clinical summary heatmap | Score matrix × patients | Annotated heatmap |

All significant hits across analyses saved to `clinical_statistics/all_significant_hits.csv`.  
Row and axis labels carry `[source_cell→sink_cell]` context throughout.

> **Note on statistical power:** With n=14 patients the analyses are exploratory.  
> p-values are reported as-is. Do not over-interpret marginal p=0.04–0.10 findings  
> without independent replication.


### 7.1 Configuration

In [ ]:
from cohort_statistics import StatConfig, build_score_matrix, run_all_clinical_analyses

STAT_CFG = StatConfig(
    base_out_dir = COHORT_OUT,
    min_score    = 0.0,     # set > 0 to exclude low-scoring metabolites
    p_thresh     = 0.05,
    top_n        = 20,
    verbose      = True,
)

STAT_OUT = os.path.join(COHORT_OUT, "clinical_statistics")
os.makedirs(STAT_OUT, exist_ok=True)
print(f"Output: {STAT_OUT}")


### 7.2 Build score matrix

In [ ]:
score_matrix = build_score_matrix(patient_results)
print(f"Score matrix: {score_matrix.shape[0]} metabolites × {score_matrix.shape[1]} patients")
print(f"Patients: {list(score_matrix.columns)}")
score_matrix.head()


### 7.3 Run all clinical analyses

In [ ]:
# Passing patient_results builds the cell-type map automatically.
# All plots and CSVs are written to COHORT_OUT/clinical_statistics/
clinical_results = run_all_clinical_analyses(
    score_matrix    = score_matrix,
    cfg             = STAT_CFG,
    patient_results = patient_results,
)
print("\nKeys returned:", list(clinical_results.keys()))


### 7.4 Summary: significant hits

In [ ]:
import glob

# Print top hits per analysis
for name, df in clinical_results.items():
    if not isinstance(df, pd.DataFrame) or df.empty:
        continue
    if "significant" not in df.columns:
        continue
    sig = df[df["significant"]]
    if sig.empty:
        print(f"  {name:15s}: no significant hits")
        continue
    n = len(sig)
    top_met = sig.iloc[0]["metabolite"] if "metabolite" in sig.columns else "?"
    print(f"  {name:15s}: {n:3d} significant  (top: {top_met})")

# Load the combined CSV if it exists
sig_csv = os.path.join(STAT_OUT, "all_significant_hits.csv")
if os.path.exists(sig_csv):
    all_sig = pd.read_csv(sig_csv)
    print(f"\nTotal across all analyses: {len(all_sig)} significant hits")
    print(all_sig.groupby("analysis")["metabolite"].count().to_string())


### 7.5 eGFR correlation results

In [ ]:
from IPython.display import Image, display

egfr_dir = os.path.join(STAT_OUT, "egfr_correlation")
figs = sorted(glob.glob(os.path.join(egfr_dir, "*.png")))
print(f"{len(figs)} eGFR figures:")
for fp in figs:
    print(f"  {os.path.basename(fp)}")

# Show the bar chart and cell-type strip (most informative)
for fname in ["egfr_bar.png", "egfr_celltype_strip.png"]:
    fp = os.path.join(egfr_dir, fname)
    if os.path.exists(fp):
        print(f"\n{fname}")
        display(Image(fp, width=1100))


### 7.6 Fibrosis association results

In [ ]:
fib_dir = os.path.join(STAT_OUT, "fibrosis_association")
for fname in ["fibrosis_bar.png", "fibrosis_tertile_boxplots.png"]:
    fp = os.path.join(fib_dir, fname)
    if os.path.exists(fp):
        print(fname)
        display(Image(fp, width=1100))


### 7.7 Binary variable volcano plots (sex / hypertension / diabetes)

In [ ]:
for subdir in ["sex_association", "hypertension_association", "diabetes_association"]:
    vdir = os.path.join(STAT_OUT, subdir)
    for fp in sorted(glob.glob(os.path.join(vdir, "volcano_*.png"))):
        print(os.path.basename(fp))
        display(Image(fp, width=900))


### 7.8 Multi-variable regression

In [ ]:
reg_dir = os.path.join(STAT_OUT, "regression")
for fname in ["regression_beta_heatmap.png", "regression_results.csv"]:
    fp = os.path.join(reg_dir, fname)
    if not os.path.exists(fp): continue
    if fname.endswith(".png"):
        print(fname)
        display(Image(fp, width=1100))
    else:
        reg_df = pd.read_csv(fp)
        print(f"\nRegression results ({len(reg_df)} rows):")
        display(reg_df.head(15))


### 7.9 Clinical summary heatmap

In [ ]:
hm = os.path.join(STAT_OUT, "clinical_summary_heatmap.png")
if os.path.exists(hm):
    display(Image(hm, width=1300))
else:
    print("Heatmap not found — check for errors above")


### 7.10 eGFR correlation table

In [ ]:
egfr_df = clinical_results.get("egfr", pd.DataFrame())
if not egfr_df.empty:
    cols = [c for c in ["metabolite","exchange_axis","rho","p_value","significant"]
            if c in egfr_df.columns]
    print(f"Top 20 eGFR-correlated metabolites (|rho| ranked):")
    display(egfr_df[cols].head(20))


## 6. Save summary

In [ ]:
rows = []
for (a,b), df in comparison_results.items():
    if df.empty: continue
    t = df.nlargest(20,"evidence_score").copy()
    t["comparison"] = f"{a}_vs_{b}"
    rows.append(t)
if rows:
    pd.concat(rows, ignore_index=True).to_csv(
        os.path.join(COHORT_OUT,"top_findings_wceg.csv"), index=False)
    print("Saved top_findings_wceg.csv")

# Folder summary
def _ls(p):
    if not os.path.exists(p): return
    entries = sorted(os.listdir(p))
    for e in entries:
        full = os.path.join(p,e)
        if os.path.isdir(full):
            n = len(os.listdir(full))
            print(f"  📁 {e}/  ({n} files)")
        else:
            print(f"     {e}  ({os.path.getsize(full)//1024} KB)")

print(f"\n{COHORT_OUT}")
_ls(COHORT_OUT)


## 8. Intracellular flux analysis

Complements the exchange-flux pipeline. Exchange fluxes answer *who trades what
with spatially proximal neighbours*; intracellular fluxes answer *what each cell
type is doing internally* (OXPHOS vs glycolysis, biosynthesis load, etc.) — and
let us mechanistically explain the exchange interactions.

**Caveats (read before interpreting):**
- FBA internal fluxes are **not unique** (alternate optima). Interpret at the
  pathway / aggregate / ratio level, not per single reaction value.
- Fluxes scale with biomass — the loader normalises by `Community_Biomass`.
- `COST_SINK` / `exchange_cost` bookkeeping rows are filtered out.

Set `INTRACELLULAR_FOLDER` to the sub-folder holding the intracellular CSVs
(sibling of `exchange_fluxes/` inside each `Region_*`).

In [ ]:
from intracellular import (
    IntracellularConfig, run_intracellular_analysis,
    find_enriched_reactions, compare_reactions_between_conditions,
    build_celltype_reaction_matrix, link_exchange_to_intracellular,
    build_spatial_flux_frame,
    load_reaction_subsystems, build_subsystem_activity_table,
    find_enriched_subsystems, build_subsystem_celltype_matrix,
    build_subsystem_prevalence_matrix, build_spatial_subsystem_frame,
    compare_subsystems_between_conditions,
    compare_celltype_subsystems_between_conditions,
    build_subsystem_differential_matrices,
    build_metabolite_activity_table, find_enriched_metabolite_activity,
    build_metabolite_activity_celltype_matrix,
    build_spatial_metabolite_activity_frame,
    compare_metabolite_activity_between_conditions,
    compare_celltype_metabolite_activity_between_conditions,
    build_metabolite_activity_differential_matrices,
    build_metabolite_pathway_allocation_table,
    compute_pathway_allocation, rank_metabolites_by_allocation_shift,
)
from intracellular_plots import (
    plot_enrichment_bar, plot_celltype_reaction_heatmap,
    plot_mechanistic_links, plot_condition_volcano,
    plot_condition_top_reactions, plot_spatial_reaction, plot_spatial_pathway,
    plot_subsystem_enrichment_dot, plot_subsystem_heatmap,
    plot_subsystem_signature_bars, plot_spatial_subsystem,
    plot_subsystem_differential_heatmap,
    plot_metabolite_pathway_allocation, plot_allocation_shift_ranking,
)

# Folder (per Region_*) that holds the intracellular flux files.
INTRACELLULAR_FOLDER = "intracellular_fluxes"

# Reaction -> Subsystem lookup produced ONCE by
# model_building/extract_reaction_subsystems.m (reads the real `subSystems`
# field off the REFERENCE Recon3D model - the same modelO2Fixed every
# cell-type/community model is GIMME-pruned from - no FBA, no solver needed).
# Using the reference model instead of the pruned per-cell models is
# deliberate: GIMME keeps only each cell type's active subset, so the
# reference is the single complete, generalizable annotation source.
# Edit modelPath/modelVarName in the .m script and run it once, then
# re-run this cell. Pathway-level cells below degrade gracefully if missing.
REACTION_SUBSYSTEMS_CSV = os.path.join(DATA_ROOT, "reaction_subsystems.csv")

SUBSYSTEM_MAP = {}
if os.path.exists(REACTION_SUBSYSTEMS_CSV):
    SUBSYSTEM_MAP = load_reaction_subsystems(REACTION_SUBSYSTEMS_CSV)
    print(f"Loaded subsystem annotations for {len(SUBSYSTEM_MAP)} reactions "
          f"from {REACTION_SUBSYSTEMS_CSV}")
else:
    print(f"No subsystem map found at {REACTION_SUBSYSTEMS_CSV} - "
          "pathway-level cells (8.1c, cohort subsystem comparison, spatial "
          "pathway maps) will be skipped until you run "
          "model_building/extract_reaction_subsystems.m.")

# All intracellular figures are also written here (like the rest of the pipeline).
IC_PLOTS = os.path.join(COHORT_OUT, "intracellular_plots")
os.makedirs(IC_PLOTS, exist_ok=True)

# --- Quick single-patient run -------------------------------------------------
# base_dir / condition / region mirror the exchange pipeline's layout exactly.
from run_cohort_pipeline import COHORT_METADATA, _condition_label, _patient_data_dir
from run_cohort_pipeline import CohortConfig

_cfg   = CohortConfig(data_root=DATA_ROOT, base_out_dir=COHORT_OUT, regions=REGIONS)
_meta0 = COHORT_METADATA[0]                       # first patient as a demo
PID0   = _meta0["id"]
COND0  = _condition_label(_meta0)

ic_cfg = IntracellularConfig(
    base_dir             = _patient_data_dir(_cfg, PID0),
    conditions           = [COND0],
    regions              = REGIONS,
    grid_pattern         = _cfg.grid_pattern,
    intracellular_folder = INTRACELLULAR_FOLDER,
    normalize_by_biomass = True,
    p_threshold          = 0.05,
)

ic_res = run_intracellular_analysis(ic_cfg, verbose=True)

print(f"\nTop cell-type-enriched reactions for {PID0} ({COND0}):")
display(ic_res.enriched_df.head(20))

### 8.1 Cell-type metabolic phenotype (reaction × cell-type heatmap)

In [ ]:
# 8.1a  Which reactions are most cell-type-specific?
fig = plot_enrichment_bar(
    ic_res.enriched_df, top_n=20,
    out_path=os.path.join(IC_PLOTS, f"{PID0}_enrichment_bar.png"),
    title=f"Most cell-type-specific reactions - {PID0} ({COND0})")
plt.show()

# 8.1b  Metabolic phenotype heatmap.
# Reactions are z-scored ACROSS cell types so specialisation is visible even
# when absolute fluxes are tiny (a raw heatmap of near-zero values reads blank).
# Grey = reaction not observed with nonzero flux in that cell type.
fig = plot_celltype_reaction_heatmap(
    ic_res.enriched_df, ic_res.celltype_matrix, top_n=30, row_zscore=True,
    out_path=os.path.join(IC_PLOTS, f"{PID0}_phenotype_heatmap.png"),
    title=f"Intracellular flux phenotype - {PID0} ({COND0})")
plt.show()

### 8.1c Pathway-level phenotype (real model subsystems)

Individual-reaction fluxes from FBA are not unique (alternate optima), so a
single reaction can be a noisy readout. Summing every reaction in a
**subsystem** (the model's real pathway annotation, e.g. "Oxidative
phosphorylation", "Glycolysis/gluconeogenesis") cancels most of that
routing noise and is the recommended default view of cell-type metabolic
phenotype. Requires `REACTION_SUBSYSTEMS_CSV` from cell 8.0 above.

In [ ]:
if SUBSYSTEM_MAP:
    subsystem_table = build_subsystem_activity_table(ic_cfg, SUBSYSTEM_MAP)
    enriched_subsystems = find_enriched_subsystems(subsystem_table)
    subsystem_matrix = build_subsystem_celltype_matrix(subsystem_table)
    subsystem_prevalence = build_subsystem_prevalence_matrix(subsystem_table)

    n_sig = int(enriched_subsystems["Significant"].sum()) if not enriched_subsystems.empty else 0
    print(f"Subsystems tested: {len(enriched_subsystems)}  |  "
          f"cell-type-enriched (p<0.05): {n_sig}")
    display(enriched_subsystems.head(15))

    # (a) the flagship figure: cell-type-specific pathway enrichment
    fig = plot_subsystem_enrichment_dot(
        enriched_subsystems, subsystem_matrix, subsystem_prevalence, top_n=25,
        out_path=os.path.join(IC_PLOTS, f"{PID0}_subsystem_enrichment_dot.png"),
        title=f"Cell-type-specific pathway enrichment — {PID0} ({COND0})")
    plt.show()

    # (b) pathway-level phenotype heatmap (cleaner than the reaction-level one)
    fig = plot_subsystem_heatmap(
        enriched_subsystems, subsystem_matrix, top_n=25,
        out_path=os.path.join(IC_PLOTS, f"{PID0}_subsystem_heatmap.png"),
        title=f"Pathway-level metabolic phenotype — {PID0} ({COND0})")
    plt.show()

    # (c) per-cell-type metabolic signatures (small multiples)
    fig = plot_subsystem_signature_bars(
        subsystem_matrix, top_n=6,
        out_path=os.path.join(IC_PLOTS, f"{PID0}_subsystem_signatures.png"))
    plt.show()
else:
    subsystem_table = pd.DataFrame()
    print("SUBSYSTEM_MAP is empty - skipping pathway-level figures. "
          "Run model_building/extract_reaction_subsystems.m and set "
          "REACTION_SUBSYSTEMS_CSV in cell 8.0.")

### 8.2 Cross-link: explain exchange interactions with internal reactions

For each exchange best-pair (`source secretes met → sink uptakes met`), surface
the internal reactions that **produce** the metabolite in the source cell type
and **consume** it in the sink — the mechanistic *why* behind the coupling.
Exchange best-pairs are read from the patient's `metabolite_summary.csv`.

In [ ]:
ms_path = os.path.join(COHORT_OUT, PID0, "metabolite_summary.csv")
exchange_best_pairs = {}
if os.path.exists(ms_path):
    ms = pd.read_csv(ms_path)
    for _, r in ms.iterrows():
        src, snk = r.get("Source"), r.get("Sink")
        if pd.notna(src) and pd.notna(snk):
            exchange_best_pairs[str(r["Metabolite"])] = (str(src), str(snk))

# Merge the per-region metabolite indices for this patient.
merged_idx = {}
for key, midx in ic_res.per_region_met_index.items():
    for met, d in midx.items():
        merged_idx.setdefault(met, {})
        for ct, roles in d.items():
            slot = merged_idx[met].setdefault(ct, {"produced": [], "consumed": []})
            slot["produced"] += roles.get("produced", [])
            slot["consumed"] += roles.get("consumed", [])

link_df = link_exchange_to_intracellular(merged_idx, exchange_best_pairs, top_k=3)
print(f"Exchange interactions mechanistically linked: "
      f"{link_df['Metabolite'].nunique() if not link_df.empty else 0}")
display(link_df.head(30))

In [ ]:
# 8.2 figure - the mechanistic 'why' behind the top exchange interactions.
# Left (cool) = reactions consuming the metabolite in the SINK cell type;
# right (warm) = reactions producing it in the SOURCE cell type.
if not link_df.empty:
    top_mets = (link_df.groupby("Metabolite")["TotalFlux"].sum()
                       .sort_values(ascending=False).head(6).index.tolist())
    fig = plot_mechanistic_links(
        link_df, metabolites=top_mets, max_metabolites=6,
        out_path=os.path.join(IC_PLOTS, f"{PID0}_mechanistic_links.png"))
    plt.show()
else:
    print("No exchange interactions could be linked - check metabolite_summary.csv "
          "and that metabolite names match between exchange and intracellular files.")

### 8.3 Cohort-wide: cell-type enrichment + between-group reaction comparison

Aggregate intracellular fluxes across all patients, then (a) find reactions that
differ across cell types cohort-wide, and (b) compare the two clinical groups
per reaction (Mann-Whitney + log2FC). CSVs are written to
`intracellular_cohort/`.

In [ ]:
from intracellular import aggregate_intracellular_fluxes
from core import load_region_data  # same loader intracellular.py uses

IC_OUT = os.path.join(COHORT_OUT, "intracellular_cohort")
os.makedirs(IC_OUT, exist_ok=True)

# Key per_region fluxes by "<Sample_group>_R<patient_id>" so the condition
# comparison can split on "_R" and pool by clinical group across patients.
cohort_per_region = {}
cohort_equations  = {}
groups_seen = []

for meta in COHORT_METADATA:
    pid  = meta["id"]
    cond = _condition_label(meta)              # e.g. "Sample_DKD"
    if cond.rsplit("_", 1)[-1] not in groups_seen:
        groups_seen.append(cond.rsplit("_", 1)[-1])
    base_dir = _patient_data_dir(_cfg, pid)
    for r in REGIONS:
        try:
            grids, meta_df, flux_dir = load_region_data(
                base_dir, cond, r, _cfg.grid_pattern, INTRACELLULAR_FOLDER)
        except Exception:
            continue
        rxn, eqs, _midx, nf, nr = aggregate_intracellular_fluxes(
            grids, meta_df, flux_dir, normalize_by_biomass=True)
        if rxn:
            cohort_per_region[f"{cond}_R{pid}"] = rxn
            cohort_equations.update(eqs)

print(f"Patients with intracellular data: {len(cohort_per_region)} region-blocks")

# (a) cohort-wide cell-type enrichment
from collections import defaultdict
pooled = defaultdict(lambda: defaultdict(list))
for rxn in cohort_per_region.values():
    for base, ctd in rxn.items():
        for ct, fx in ctd.items():
            pooled[base][ct].extend(fx)
pooled = {b: dict(d) for b, d in pooled.items()}

enriched_cohort = find_enriched_reactions(pooled, 0.05, cohort_equations)
enriched_cohort.to_csv(os.path.join(IC_OUT, "intracellular_celltype_enrichment.csv"), index=False)
print(f"Cell-type-enriched reactions (cohort): "
      f"{int(enriched_cohort['Significant'].sum()) if not enriched_cohort.empty else 0}")
display(enriched_cohort.head(15))

# (b) between-group comparison (first two clinical groups encountered)
if len(groups_seen) >= 2:
    two = [f"Sample_{groups_seen[0]}", f"Sample_{groups_seen[1]}"]
    cmp_df = compare_reactions_between_conditions(cohort_per_region, two)
    cmp_df.to_csv(os.path.join(IC_OUT, "intracellular_group_comparison.csv"), index=False)
    print(f"\n{two[0]} vs {two[1]} — reactions compared: {len(cmp_df)}")
    display(cmp_df.head(20))
else:
    print("Only one clinical group present — skipping between-group comparison.")

In [ ]:
# 8.3 figures - differential intracellular flux between clinical groups.
if len(groups_seen) >= 2 and not cmp_df.empty:
    c0, c1 = two  # e.g. Sample_DKD, Sample_Reference
    fig = plot_condition_volcano(
        cmp_df, cond0=c0, cond1=c1, alpha=0.05, label_top=12,
        out_path=os.path.join(IC_OUT, "intracellular_volcano.png"))
    plt.show()

    fig = plot_condition_top_reactions(
        cmp_df, top_n=20,
        out_path=os.path.join(IC_OUT, "intracellular_top_differential.png"),
        title=f"Top differential reactions ({c1} vs {c0})")
    plt.show()
else:
    print("Between-group figures skipped (need >= 2 clinical groups with data).")

### 8.3b Cohort-wide pathway comparison between clinical groups

In [ ]:
if SUBSYSTEM_MAP and len(groups_seen) >= 2:
    # Build the cohort-wide subsystem activity table across all patients
    # (one pass per patient over their own community files).
    frames = []
    for meta in COHORT_METADATA:
        pid, cond = meta["id"], _condition_label(meta)
        cfg_p = IntracellularConfig(
            base_dir=_patient_data_dir(_cfg, pid), conditions=[cond], regions=REGIONS,
            grid_pattern=_cfg.grid_pattern, intracellular_folder=INTRACELLULAR_FOLDER)
        t = build_subsystem_activity_table(cfg_p, SUBSYSTEM_MAP)
        if not t.empty:
            frames.append(t)
    cohort_subsystem_table = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    print(f"Cohort subsystem-activity samples: {len(cohort_subsystem_table)}")

    enriched_subsystems_cohort = find_enriched_subsystems(cohort_subsystem_table)
    enriched_subsystems_cohort.to_csv(
        os.path.join(IC_OUT, "intracellular_subsystem_enrichment.csv"), index=False)

    two_sub = [f"Sample_{groups_seen[0]}", f"Sample_{groups_seen[1]}"]
    subsystem_cmp_df = compare_subsystems_between_conditions(cohort_subsystem_table, two_sub)
    subsystem_cmp_df.to_csv(
        os.path.join(IC_OUT, "intracellular_subsystem_group_comparison.csv"), index=False)

    fig = plot_condition_volcano(
        subsystem_cmp_df, cond0=two_sub[0], cond1=two_sub[1], alpha=0.05,
        label_top=12, label_col="Subsystem",
        out_path=os.path.join(IC_OUT, "intracellular_subsystem_volcano.png"))
    plt.show()

    fig = plot_condition_top_reactions(
        subsystem_cmp_df, top_n=20, label_col="Subsystem",
        out_path=os.path.join(IC_OUT, "intracellular_subsystem_top_differential.png"),
        title=f"Top differential pathways ({two_sub[1]} vs {two_sub[0]})")
    plt.show()
else:
    print("Skipped (need SUBSYSTEM_MAP and >= 2 clinical groups).")

### 8.3c Differential pathway activity, stratified by cell type

8.3b compares pathways between clinical groups pooling across all cell
types, so a hit only tells you a pathway shifted *somewhere*. This keeps
cell type as a second axis: for every (pathway, cell type) pair, is it
significantly different between groups, and in which direction? This is
the pathway-flux-sum analogue of the exchange side's 9.3 differential
heatmap - same statistic (Mann-Whitney + log2FC), same significance-star
convention, one level up at pathway resolution instead of per-metabolite.

In [ ]:
if SUBSYSTEM_MAP and len(groups_seen) >= 2:
    diff_pathway_df = compare_celltype_subsystems_between_conditions(
        cohort_subsystem_table, two_sub)

    if diff_pathway_df.empty:
        print("Not enough per-(pathway, cell type) samples for a differential test.")
    else:
        diff_pathway_df.to_csv(
            os.path.join(IC_OUT, "intracellular_subsystem_celltype_differential.csv"),
            index=False)
        print(f"Top differential (pathway, cell type) pairs "
              f"({two_sub[0]} vs {two_sub[1]}):")
        display(diff_pathway_df.head(10))

        l2fc_mat, pval_mat = build_subsystem_differential_matrices(diff_pathway_df)
        fig = plot_subsystem_differential_heatmap(
            l2fc_mat, pval_mat, two_sub[0], two_sub[1], alpha=0.05,
            out_path=os.path.join(IC_OUT, "intracellular_subsystem_celltype_differential.png"))
        plt.show()
else:
    print("Skipped (need SUBSYSTEM_MAP and >= 2 clinical groups).")

### 8.4 Spatial intracellular flux maps

The exchange pipeline draws streamlines of metabolite *flow between* cells.
The intracellular analogue is a per-cell **metabolic activity map**: each
cell is placed at its `(px_x, px_y)` and coloured by its own internal flux,
averaged across every community it participated in. This reveals spatial
heterogeneity of metabolism (e.g. an OXPHOS-high tubular region vs a
glycolytic front) that boundary exchanges alone cannot show.

- Single-reaction maps use a **diverging** scale centred at 0 (signed flux).
- Pathway maps use the model's real **subsystems** (via `SUBSYSTEM_MAP`) and
  a **sequential** magnitude scale. If no subsystem map is loaded, this
  falls back to single-reaction maps only.

In [ ]:
# Single most cell-type-specific reaction, mapped in tissue space
# (annotated: cell-type reference panel + patient/condition context + top-hotspot ring).
top_enriched_rxns = ic_res.enriched_df.head(12)["Reaction"].tolist()
spatial = build_spatial_flux_frame(ic_cfg, reactions=top_enriched_rxns, normalize_by_biomass=True)
print(f"Spatial frame: {spatial.shape[0]} cell-reaction rows, "
      f"{spatial['barcode'].nunique() if not spatial.empty else 0} cells, "
      f"{spatial['reaction'].nunique() if not spatial.empty else 0} reactions")

if not spatial.empty:
    rxn0 = top_enriched_rxns[0]
    fig = plot_spatial_reaction(
        spatial, rxn0, patient_id=PID0, condition=COND0,
        out_path=os.path.join(IC_PLOTS, f"{PID0}_spatial_{rxn0}.png"))
    plt.show()
else:
    print("Empty spatial frame - check INTRACELLULAR_FOLDER / file naming.")

# Pathway-level spatial maps, driven by the model's real subsystems
# (top subsystems by cell-type enrichment - no hardcoded pathway list).
if SUBSYSTEM_MAP and not subsystem_table.empty:
    spatial_pathway_frame = build_spatial_subsystem_frame(subsystem_table)
    top_subs_spatial = enriched_subsystems.head(4)["Subsystem"].tolist()
    for sub in top_subs_spatial:
        safe = "".join(c if c.isalnum() else "_" for c in sub)[:40]
        fig = plot_spatial_subsystem(
            spatial_pathway_frame, sub, patient_id=PID0, condition=COND0,
            out_path=os.path.join(IC_PLOTS, f"{PID0}_spatial_subsystem_{safe}.png"))
        plt.show()
else:
    print("No subsystem map loaded - skipping pathway-level spatial maps "
          "(see cell 8.0 / model_building/extract_reaction_subsystems.m).")

### 8.5 Metabolite flux-sum: per-metabolite turnover (no subsystem map needed)

Pathway-flux-sum (8.1c/8.3b/8.3c) groups reactions by the reference model's
CURATED CATEGORY ("TCA cycle", "Glycolysis", ...). Metabolite-flux-sum
instead groups reactions by a shared NETWORK NODE: for a given metabolite
(e.g. atp, nadh), sum |flux| over every reaction that has it as EITHER a
substrate or product, regardless of which pathway(s) those reactions belong
to. A metabolite like ATP is used by reactions scattered across dozens of
pathways, so this is a genuinely different grouping axis, not a finer/
coarser version of the pathway view. Unlike the pathway view, this needs
**no MATLAB/Recon3D subsystem lookup at all** - metabolite names are parsed
directly from each row's own reaction equation.

In [ ]:
met_table = build_metabolite_activity_table(ic_cfg)
print(f"Metabolite-activity samples: {len(met_table)}  |  "
      f"{met_table['metabolite'].nunique() if not met_table.empty else 0} metabolites")

if not met_table.empty:
    enriched_metabolites = find_enriched_metabolite_activity(met_table)
    met_celltype_matrix = build_metabolite_activity_celltype_matrix(met_table)
    n_sig = int(enriched_metabolites["Significant"].sum()) if not enriched_metabolites.empty else 0
    print(f"Cell-type-enriched metabolites (p<0.05): {n_sig}")
    display(enriched_metabolites.head(15))

    # Reuses the pathway phenotype heatmap renderer via label_col="Metabolite".
    fig = plot_subsystem_heatmap(
        enriched_metabolites, met_celltype_matrix, top_n=25, label_col="Metabolite",
        title=f"Metabolite-turnover phenotype — {PID0} ({COND0})",
        out_path=os.path.join(IC_PLOTS, f"{PID0}_metabolite_heatmap.png"))
    plt.show()
else:
    print("Empty metabolite-activity table - check INTRACELLULAR_FOLDER / file naming.")

### 8.6 Combined: pathway x metabolite flux ALLOCATION (emergent insight)

Neither axis alone can answer "is this metabolite's flux being ROUTED
differently between conditions" - pathway-flux-sum pools every metabolite
together within a pathway; metabolite-flux-sum pools every pathway together
for one metabolite. Crossing them (`build_metabolite_pathway_allocation_
table`) gives, for each metabolite, what FRACTION of its total turnover
comes from each pathway. A metabolite's total activity can stay flat
between two conditions while its pathway allocation reorganises completely
(e.g. ATP unchanged in total, but the fraction produced by OXPHOS vs
glycolysis flips) - `rank_metabolites_by_allocation_shift` finds exactly
these cases via a total-variation-distance score (0 = identical pathway
usage, 1 = completely disjoint), which a magnitude-only differential test
would miss entirely.

In [ ]:
# Single-patient allocation table (informational - a shift needs >= 2 conditions
# to compare against, so the ranking below is computed cohort-wide instead).
if SUBSYSTEM_MAP:
    alloc_table = build_metabolite_pathway_allocation_table(ic_cfg, SUBSYSTEM_MAP)
    print(f"Allocation samples for {PID0} ({COND0}): {len(alloc_table)}")
else:
    alloc_table = pd.DataFrame()
    print("SUBSYSTEM_MAP is empty - skipping (needs the same Recon3D lookup as 8.1c/8.3b/8.3c).")

# Cohort-wide allocation-shift ranking (needs >= 2 conditions), reusing the same
# per-patient loop pattern as 8.3b/8.3c.
if SUBSYSTEM_MAP and len(groups_seen) >= 2:
    alloc_frames = []
    for meta in COHORT_METADATA:
        pid, cond = meta["id"], _condition_label(meta)
        cfg_p = IntracellularConfig(
            base_dir=_patient_data_dir(_cfg, pid), conditions=[cond], regions=REGIONS,
            grid_pattern=_cfg.grid_pattern, intracellular_folder=INTRACELLULAR_FOLDER)
        t = build_metabolite_pathway_allocation_table(cfg_p, SUBSYSTEM_MAP)
        if not t.empty:
            alloc_frames.append(t)
    cohort_alloc_table = pd.concat(alloc_frames, ignore_index=True) if alloc_frames else pd.DataFrame()
    print(f"Cohort allocation samples: {len(cohort_alloc_table)}")

    shift_df = rank_metabolites_by_allocation_shift(cohort_alloc_table, two_sub, min_pathways=2)
    shift_df.to_csv(os.path.join(IC_OUT, "metabolite_allocation_shift.csv"), index=False)
    print(f"\nMost-rerouted metabolites ({two_sub[0]} vs {two_sub[1]}):")
    display(shift_df.head(10))

    if not shift_df.empty:
        fig = plot_allocation_shift_ranking(
            shift_df, two_sub[0], two_sub[1], top_n=15,
            out_path=os.path.join(IC_OUT, "metabolite_allocation_shift_ranking.png"))
        plt.show()

        # Detail view for the single most-rerouted metabolite.
        top_met = shift_df.iloc[0]["Metabolite"]
        alloc_frac = compute_pathway_allocation(cohort_alloc_table, group_cols=["condition"])
        fig = plot_metabolite_pathway_allocation(
            alloc_frac, top_met, two_sub,
            out_path=os.path.join(IC_OUT, f"metabolite_allocation_{top_met}.png"))
        plt.show()
elif SUBSYSTEM_MAP:
    print("Skipped cohort-wide ranking (need >= 2 clinical groups). "
          "Single-patient allocation table (alloc_table) is still available above.")

## 9. Exchange flux: per-cell-type secretion/uptake analysis

Existing exchange-side figures either collapse across metabolites
(`plot_flux_balance_comparison`: total produced/consumed per cell type,
metabolite-agnostic) or collapse across cell types (`plot_top_metabolites_bar`,
WCEG: one metabolite tied to its single dominant source/sink pair). This
section keeps **both dimensions** - cell type AND metabolite - to answer:

- **9.1** For one cell type, what are its top secreted / top taken-up
  metabolites, and how does that profile shift DKD vs HKD vs Control?
- **9.2** Across the whole cohort, which individual metabolites (not
  aggregated into one score) dominate secretion / uptake, broken out per
  cell type?

Reuses `patient_results` loaded in section 3.1 and the same clinical-group
colour palette as the WCEG plots (green=Control, orange=HKD, red=DKD).

In [ ]:
from exchange_celltype_analysis import (
    build_secretion_uptake_table, top_metabolites_for_celltype,
    celltype_metabolite_by_group, rank_celltypes_by_activity,
    top_metabolites_overall, build_celltype_metabolite_matrix,
    compare_celltype_metabolite_between_conditions, build_differential_matrices,
)
from exchange_celltype_plots import (
    plot_celltype_secretion_uptake, plot_top_metabolites_celltype_heatmap,
    plot_differential_celltype_heatmap,
)

EXCHANGE_PLOTS = os.path.join(COHORT_OUT, "exchange_celltype_plots")
os.makedirs(EXCHANGE_PLOTS, exist_ok=True)

# Clinical group display order (must match whatever groups exist in COHORT_METADATA).
GROUP_ORDER = ["Control", "HKD", "DKD"]

su_table = build_secretion_uptake_table(patient_results)
print(f"Secretion/uptake samples: {len(su_table)}  |  "
      f"{su_table['metabolite'].nunique()} metabolites, "
      f"{su_table['cell_type'].nunique()} cell types, "
      f"groups: {sorted(su_table['group'].unique())}")

### 9.1 Per-cell-type: top secreted/taken-up metabolites across conditions

`TOP_CELLTYPES_N` picks the most metabolically active cell types (by total
|flux|) to feature by default - edit `FEATURED_CELLTYPES` to target specific
ones instead. One figure per cell type; also saved to `exchange_celltype_plots/`.

In [ ]:
TOP_CELLTYPES_N   = 4    # how many cell types to auto-feature
TOP_METABOLITES_N = 10   # top-N secreted / top-N taken-up metabolites per cell type

FEATURED_CELLTYPES = rank_celltypes_by_activity(su_table, top_n=TOP_CELLTYPES_N)
print(f"Featured cell types (by total exchange activity): {FEATURED_CELLTYPES}")

for ct in FEATURED_CELLTYPES:
    mat_sec = celltype_metabolite_by_group(su_table, ct, "secretion", top_n=TOP_METABOLITES_N)
    mat_upt = celltype_metabolite_by_group(su_table, ct, "uptake",    top_n=TOP_METABOLITES_N)
    if mat_sec.empty and mat_upt.empty:
        print(f"{ct}: no data, skipping")
        continue
    safe = "".join(c if c.isalnum() else "_" for c in ct)[:40]
    fig = plot_celltype_secretion_uptake(
        mat_sec, mat_upt, ct, group_order=GROUP_ORDER,
        out_path=os.path.join(EXCHANGE_PLOTS, f"secretion_uptake_{safe}.png"))
    plt.show()

### 9.2 Cohort-wide: top individual metabolites by secretion/uptake, across cell types

Unlike 9.1 (one cell type at a time, ranked within it), this ranks
metabolites **globally** by aggregate |flux| and shows each one's full
breakdown across every cell type - so a metabolite that is high everywhere
looks different from one that is high only in a single cell type.

In [ ]:
TOP_METABOLITES_OVERALL = 20

mat_sec_all = build_celltype_metabolite_matrix(su_table, "secretion", top_n=TOP_METABOLITES_OVERALL)
mat_upt_all = build_celltype_metabolite_matrix(su_table, "uptake",    top_n=TOP_METABOLITES_OVERALL)

fig = plot_top_metabolites_celltype_heatmap(
    mat_sec_all, mat_upt_all,
    out_path=os.path.join(EXCHANGE_PLOTS, "top_metabolites_by_celltype_heatmap.png"))
plt.show()

# Optional: the same view restricted to one clinical group at a time, to see
# whether the dominant metabolites/cell-type pattern itself shifts with disease.
for grp in GROUP_ORDER:
    if grp not in su_table["group"].unique():
        continue
    ms = build_celltype_metabolite_matrix(su_table, "secretion", top_n=TOP_METABOLITES_OVERALL, group=grp)
    mu = build_celltype_metabolite_matrix(su_table, "uptake",    top_n=TOP_METABOLITES_OVERALL, group=grp)
    if ms.empty and mu.empty:
        continue
    fig = plot_top_metabolites_celltype_heatmap(
        ms, mu, title_suffix=f" ({grp})",
        out_path=os.path.join(EXCHANGE_PLOTS, f"top_metabolites_by_celltype_heatmap_{grp}.png"))
    plt.show()

### 9.3 Differential flux analysis: which (metabolite, cell type) pairs shift between conditions

The heatmaps above show magnitude; this goes one step further and tests, for
every (metabolite, cell type) pair, whether secretion/uptake actually
*differs* between two clinical groups (Mann-Whitney U + log2 fold-change) -
so you can see not just that a metabolite's exchange is high, but that it is
specifically higher/lower in disease, and in which cell type.

In [ ]:
DIFF_GROUP_A, DIFF_GROUP_B = "Control", "DKD"   # edit to the groups present in your cohort
DIFF_TOP_N = 20

for role in ["secretion", "uptake"]:
    diff_df = compare_celltype_metabolite_between_conditions(
        su_table, role, DIFF_GROUP_A, DIFF_GROUP_B, top_n=DIFF_TOP_N)
    if diff_df.empty:
        print(f"{role}: not enough data for {DIFF_GROUP_A} vs {DIFF_GROUP_B} - skipping.")
        continue

    diff_df.to_csv(
        os.path.join(EXCHANGE_PLOTS, f"differential_{role}_{DIFF_GROUP_A}_vs_{DIFF_GROUP_B}.csv"),
        index=False)
    print(f"\n{role}: top differential (metabolite, cell type) pairs "
          f"({DIFF_GROUP_A} vs {DIFF_GROUP_B}):")
    display(diff_df.head(10))

    l2fc_mat, pval_mat = build_differential_matrices(diff_df)
    fig = plot_differential_celltype_heatmap(
        l2fc_mat, pval_mat, role, DIFF_GROUP_A, DIFF_GROUP_B, alpha=0.05,
        out_path=os.path.join(EXCHANGE_PLOTS, f"differential_{role}_heatmap.png"))
    plt.show()

## 10. Intracellular flux vs clinical covariates (eGFR, Fibrosis)

Mirrors the exchange-side eGFR/Fibrosis correlation (`cohort_statistics.py`:
Spearman rho, nominal p<0.05, matching `egfr_df`'s convention) but for
intracellular pathway activity, at two resolutions:

- **10.1 Aggregate** - one value per (patient, subsystem), pooled across
  cell types - directly comparable to the exchange-side analysis.
- **10.2 Per-cell-type** - one value per (patient, subsystem, cell type),
  correlated independently per cell type. This can surface a clinical
  association that is invisible in the pooled/aggregate view (e.g. "OXPHOS in
  PT tracks eGFR" even if whole-tissue OXPHOS does not).

Requires `SUBSYSTEM_MAP` from section 8.0. An additional BH-FDR-adjusted
p-value (`padj`, threshold 0.20 - matching this notebook's existing
`PADJ_THRESHOLD` convention) is reported alongside the nominal p-value,
since a stricter FDR is generally unusable at this cohort's sample size.

In [ ]:
from intracellular_clinical import (
    build_patient_entity_matrix, build_patient_celltype_entity_matrices,
    correlate_with_clinical, correlate_celltype_with_clinical,
    plot_clinical_correlation_bar, plot_clinical_correlation_scatter,
)

CLINICAL_IC_OUT = os.path.join(COHORT_OUT, "intracellular_clinical")
os.makedirs(CLINICAL_IC_OUT, exist_ok=True)

CLINICAL_FIELDS_IC = ["eGFR", "fibrosis"]
CLINICAL_ENTITY_COL = "subsystem"   # or "reaction" for reaction-level correlation
CLINICAL_MIN_PATIENTS = 5

if not SUBSYSTEM_MAP:
    print("SUBSYSTEM_MAP is empty - skipping section 10 "
          "(run model_building/extract_reaction_subsystems.m and set "
          "REACTION_SUBSYSTEMS_CSV in cell 8.0).")
else:
    # Build one subsystem-activity table per patient (reused from section 8.3's
    # cohort loop pattern), keyed by patient_id so per-patient pooling is exact.
    per_patient_tables = {}
    for meta in COHORT_METADATA:
        pid, cond = meta["id"], _condition_label(meta)
        cfg_p = IntracellularConfig(
            base_dir=_patient_data_dir(_cfg, pid), conditions=[cond], regions=REGIONS,
            grid_pattern=_cfg.grid_pattern, intracellular_folder=INTRACELLULAR_FOLDER)
        t = build_subsystem_activity_table(cfg_p, SUBSYSTEM_MAP)
        if not t.empty:
            per_patient_tables[pid] = t
    print(f"Patients with intracellular data for clinical correlation: {len(per_patient_tables)}")

### 10.1 Aggregate: pathway activity (pooled across cell types) vs eGFR / Fibrosis

In [ ]:
if SUBSYSTEM_MAP and per_patient_tables:
    agg_matrix = build_patient_entity_matrix(per_patient_tables, entity_col=CLINICAL_ENTITY_COL)
    print(f"Aggregate matrix: {agg_matrix.shape[0]} subsystems x {agg_matrix.shape[1]} patients")

    for field in CLINICAL_FIELDS_IC:
        corr_agg = correlate_with_clinical(agg_matrix, field, min_patients=CLINICAL_MIN_PATIENTS)
        if corr_agg.empty:
            print(f"{field}: not enough patients/data - skipping.")
            continue
        corr_agg.to_csv(os.path.join(CLINICAL_IC_OUT, f"aggregate_{field}_correlation.csv"), index=False)
        print(f"\n{field}: top aggregate pathway correlations:")
        display(corr_agg.head(10))

        fig = plot_clinical_correlation_bar(
            corr_agg, field, top_n=15,
            title=f"Aggregate pathway activity vs {field}",
            out_path=os.path.join(CLINICAL_IC_OUT, f"aggregate_{field}_bar.png"))
        plt.show()

        # Scatter for the single strongest hit.
        top_entity = corr_agg.iloc[0]["Entity"]
        fig = plot_clinical_correlation_scatter(
            agg_matrix, field, top_entity,
            out_path=os.path.join(CLINICAL_IC_OUT, f"aggregate_{field}_{top_entity}_scatter.png"))
        plt.show()

### 10.2 Per-cell-type: pathway activity vs eGFR / Fibrosis, stratified by cell type

In [ ]:
if SUBSYSTEM_MAP and per_patient_tables:
    celltype_matrices = build_patient_celltype_entity_matrices(
        per_patient_tables, entity_col=CLINICAL_ENTITY_COL)
    print(f"Cell types with enough data: {list(celltype_matrices.keys())}")

    for field in CLINICAL_FIELDS_IC:
        corr_ct = correlate_celltype_with_clinical(
            celltype_matrices, field, min_patients=CLINICAL_MIN_PATIENTS)
        if corr_ct.empty:
            print(f"{field}: not enough patients/data - skipping.")
            continue
        corr_ct.to_csv(os.path.join(CLINICAL_IC_OUT, f"celltype_{field}_correlation.csv"), index=False)
        print(f"\n{field}: top per-cell-type pathway correlations:")
        display(corr_ct.head(10))

        fig = plot_clinical_correlation_bar(
            corr_ct, field, top_n=15,
            title=f"Per-cell-type pathway activity vs {field}",
            out_path=os.path.join(CLINICAL_IC_OUT, f"celltype_{field}_bar.png"))
        plt.show()

        # Scatter for the single strongest (entity, cell_type) hit.
        top_row = corr_ct.iloc[0]
        top_entity, top_ct = top_row["Entity"], top_row["CellType"]
        fig = plot_clinical_correlation_scatter(
            celltype_matrices[top_ct], field, top_entity, cell_type=top_ct,
            out_path=os.path.join(CLINICAL_IC_OUT, f"celltype_{field}_{top_entity}_{top_ct}_scatter.png"))
        plt.show()